# Gemma 4 E4B IT — focused text-only J-lens recalibration

This notebook repairs the calibration/evaluation mismatch exposed by the SpokenCOCO pilot. It fits only layers 35 and 38 on text rendered with Gemma's official IT chat template, validates on disjoint held-out text, and publishes a frozen artifact only if it beats a pool-matched random-direction control. It never reads images, audio, SpokenCOCO captions, or multimodal test activations.


In [ ]:
# 1. Primitive bootstrap constants only
REPO_URL = "https://github.com/MechInterpreter/jacobian-lens-gemma.git"
REPO_BRANCH = "experiment/spokencoco-jspace-pilot"
REPO_DIR = "/content/jacobian-lens-gemma"
print(REPO_URL, REPO_BRANCH, REPO_DIR, sep="\n")


In [ ]:
# 2. Idempotent checkout and editable install
import os, subprocess, sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path(REPO_DIR)
    if not (repo / ".git").exists():
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
branch = subprocess.run(["git", "branch", "--show-current"], check=True, capture_output=True, text=True).stdout.strip()
commit = subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
if branch != REPO_BRANCH:
    raise RuntimeError(f"expected {REPO_BRANCH}, checked out {branch}")
import jlens
print(f"branch={branch}\ncommit={commit}\njlens={jlens.__file__}")


In [ ]:
# 3. Configuration — real fitting never starts automatically
RUN_RECALIBRATION = False  # change manually in Colab
CONFIG_PATH = "configs/gemma_text_recalibration.yaml"
DRIVE_RUNS_ROOT = "/content/drive/MyDrive/jacobian-lens-gemma/runs"
from jlens.metadata import config_fingerprint, load_config
CONFIG = load_config(CONFIG_PATH)
CONFIG_FINGERPRINT = config_fingerprint(CONFIG)
print(f"RUN_RECALIBRATION={RUN_RECALIBRATION}\nconfig={CONFIG_PATH}\nfingerprint={CONFIG_FINGERPRINT}")
print("text-only protocol:", CONFIG["recalibration"]["protocol"])


In [ ]:
# 4. Mount Drive and establish a stable resumable run directory
RUN_DIR = None
if RUN_RECALIBRATION:
    if not IN_COLAB:
        raise RuntimeError("real recalibration is intended for Colab on an NVIDIA L4")
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path(DRIVE_RUNS_ROOT) / CONFIG["recalibration"]["run_name"]
    (RUN_DIR / "artifacts").mkdir(parents=True, exist_ok=True)
    (RUN_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
    state_path = RUN_DIR / "calibration_state.json"
    if state_path.exists():
        import json
        state = json.loads(state_path.read_text())
        if state["config_fingerprint"] != CONFIG_FINGERPRINT:
            raise RuntimeError(f"{RUN_DIR} contains an incompatible calibration; choose a new run_name")
        print("run state: resuming")
    else:
        state_path.write_text(__import__("json").dumps({"config_fingerprint": CONFIG_FINGERPRINT, "protocol": CONFIG["recalibration"]["protocol"]}, indent=2))
        print("run state: starting")
    print("run directory:", RUN_DIR)
else:
    print("model load and fitting disabled; set RUN_RECALIBRATION=True manually")


In [ ]:
# 5. Authenticate, load the exact checkpoint, and verify architecture
MODEL = LOAD_INFO = ARCH = None
if RUN_RECALIBRATION:
    import torch
    from huggingface_hub import login
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if not token:
        raise RuntimeError("Add an HF_TOKEN Colab secret with Gemma access")
    login(token=token, add_to_git_credential=False)
    from jlens.gemma4 import load_gemma4, verify_architecture
    MODEL, LOAD_INFO = load_gemma4(repo_id=CONFIG["model"]["repo_id"], revision=CONFIG["model"]["revision"], dtype=torch.bfloat16, device_map="cuda", allow_model_load=True, token=token)
    ARCH = verify_architecture(MODEL, expect_n_layers=42, expect_d_model=2560, expect_vocab_size=262144)
    print(ARCH)


In [ ]:
# 6. Deterministic, disjoint text calibration and validation prompts
FIT_PROMPTS = HELDOUT_PROMPTS = None
if RUN_RECALIBRATION:
    import hashlib, json
    from jlens.examples import load_wikitext_prompts
    n_fit = CONFIG["fitting"]["n_prompts"]
    n_test = CONFIG["recalibration"]["heldout_prompts"]
    raw = load_wikitext_prompts(n_fit + n_test)
    instruction = CONFIG["recalibration"]["chat_instruction"]
    max_tokens = CONFIG["fitting"]["max_seq_len"]
    def render(passage):
        # Shorten only the passage, never right-truncate the assistant-generation suffix.
        passage = passage.strip()
        while True:
            text = MODEL.tokenizer.apply_chat_template([{"role": "user", "content": f"{instruction}\n\n{passage}"}], tokenize=False, add_generation_prompt=True)
            ids = MODEL.tokenizer(text, add_special_tokens=False)["input_ids"]
            if len(ids) + 1 <= max_tokens:
                return text
            if len(passage) <= 64:
                raise RuntimeError("chat template alone exceeds max_seq_len")
            passage = passage[:max(64, int(len(passage) * 0.85))]
    rendered = [render(text) for text in raw]
    FIT_PROMPTS, HELDOUT_PROMPTS = rendered[:n_fit], rendered[n_fit:]
    hashes = [hashlib.sha256(p.encode()).hexdigest() for p in rendered]
    if set(hashes[:n_fit]) & set(hashes[n_fit:]):
        raise RuntimeError("calibration/validation prompt leakage")
    metadata = {"source": "Salesforce/wikitext wikitext-103-raw-v1", "protocol": CONFIG["recalibration"]["protocol"], "fit_hashes": hashes[:n_fit], "heldout_hashes": hashes[n_fit:]}
    (RUN_DIR / "prompt_metadata.json").write_text(json.dumps(metadata, indent=2))
    print(f"fit={len(FIT_PROMPTS)} heldout={len(HELDOUT_PROMPTS)} leakage=0")


In [ ]:
# 7. Fit or resume the two-layer text-only lens
LENS = None
if RUN_RECALIBRATION:
    import time
    from jlens.fitting import fit
    from jlens.lens import JacobianLens
    lens_path = RUN_DIR / "artifacts" / "lens.candidate.pt"
    checkpoint = RUN_DIR / "checkpoints" / "ckpt.pt"
    if lens_path.exists():
        LENS = JacobianLens.load(str(lens_path))
        print("fit: reused completed candidate", LENS)
    else:
        started = time.perf_counter()
        LENS = fit(MODEL, FIT_PROMPTS, source_layers=CONFIG["sites"]["source_layers"], target_layer=CONFIG["sites"]["target_layer"], dim_batch=CONFIG["fitting"]["dim_batch"], max_seq_len=CONFIG["fitting"]["max_seq_len"], skip_first=CONFIG["positions"]["skip_first"], checkpoint_path=str(checkpoint), checkpoint_every=CONFIG["fitting"]["checkpoint_every"])
        LENS.save(str(lens_path))
        print(f"fit completed in {(time.perf_counter()-started)/60:.1f} minutes:", LENS)


In [ ]:
# 8. Held-out text gate against exact pool-matched random directions
VALIDATION = None
if RUN_RECALIBRATION:
    import json, os, torch
    from jlens.hooks import ActivationRecorder
    from jlens.mmpilot.jspace import build_dictionary, tensor_checksum
    from jlens.mmpilot.reconstruction import ReconstructionControlConfig, reconstruction_control_record, summarize_reconstruction_controls
    from jlens.pursuit import PursuitSettings
    settings = PursuitSettings(k=CONFIG["recalibration"]["pursuit_k"], tol_relative_residual=0.0)
    control = ReconstructionControlConfig(n_draws=CONFIG["recalibration"]["control_draws"], max_samples_per_layer=CONFIG["recalibration"]["max_control_samples_per_layer"], max_control_pool_atoms=None, require_pool_match=True, pool_ladder=())
    records = []
    for layer in CONFIG["sites"]["source_layers"]:
        dictionary = build_dictionary(LENS, layer, MODEL._lm_head.weight, device="cuda", dtype=torch.float16, build_chunk_rows=16384)
        for index, prompt in enumerate(HELDOUT_PROMPTS):
            ids = MODEL.encode(prompt, max_length=CONFIG["fitting"]["max_seq_len"])
            with ActivationRecorder(MODEL.layers, at=[layer]) as recorder:
                MODEL.forward(ids)
            activation = recorder.activations[layer][0, -1].detach().float().cpu()
            records.append(reconstruction_control_record(activation, dictionary, settings, config=control, sample_id=f"heldout-{index}", layer=layer, modality="text", split="test", activation_checksum=tensor_checksum(activation), lens_checksum="candidate"))
            print(f"L{layer} heldout {index+1}/{len(HELDOUT_PROMPTS)} complete")
        del dictionary
        torch.cuda.empty_cache()
    VALIDATION = summarize_reconstruction_controls(records, config=control, primary_layer=CONFIG["sites"]["source_layers"][-1])
    tmp = RUN_DIR / "artifacts" / "heldout_validation.json.tmp"
    final = RUN_DIR / "artifacts" / "heldout_validation.json"
    tmp.write_text(json.dumps(VALIDATION, indent=2)); os.replace(tmp, final)
    for layer, row in VALIDATION["by_layer"].items():
        print(f"L{layer}: lens={row['median_explained_fraction']:.4f} random={row['median_random_median_explained_fraction']:.4f} excess={row['median_excess_explained_fraction']:+.4f} above={row['above_random']}")


In [ ]:
# 9. Publish only a validated frozen lens
if RUN_RECALIBRATION:
    import hashlib, json, os, shutil
    required = CONFIG["recalibration"]["require_layers_above_random"]
    passing = VALIDATION["layers_above_random"]
    if len(passing) < required:
        print(f"RECALIBRATION NO-GO: only {passing} beat matched random; candidate remains unpublished")
    else:
        source = RUN_DIR / "artifacts" / "lens.candidate.pt"
        published = RUN_DIR / "artifacts" / "lens.validated.pt"
        publish_tmp = RUN_DIR / "artifacts" / "lens.validated.pt.tmp"
        shutil.copyfile(source, publish_tmp); os.replace(publish_tmp, published)
        checksum = "sha256:" + hashlib.sha256(published.read_bytes()).hexdigest()
        manifest = {"status": "validated_text_only", "lens_path": str(published), "lens_checksum": checksum, "model_revision": LOAD_INFO["model_revision"], "source_layers": LENS.source_layers, "n_prompts": LENS.n_prompts, "prompt_protocol": CONFIG["recalibration"]["protocol"], "layers_above_random": passing, "validation_path": str(RUN_DIR / "artifacts" / "heldout_validation.json")}
        (RUN_DIR / "artifacts" / "validated_lens_manifest.json").write_text(json.dumps(manifest, indent=2))
        print("VALIDATED TEXT-ONLY LENS READY")
        print(json.dumps(manifest, indent=2))
